In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [3]:
np.random.seed(0)
num_cases = 5000  
regions = ['APAC', 'EMEA', 'AMER']
client_types = ['Corporate', 'SME', 'Institutional']
stages = [
    '1_Document_Submission', 
    '2_KYC_AML_Check', 
    '3_Credit_Risk_Assessment', 
    '4_Legal_Compliance_Approval', 
    '5_Account_Activation'
]

In [4]:
data = []
start_date_base = datetime(2026, 1, 1)

In [17]:
for case_id in range(1001, 1001 + num_cases):
    region = random.choice(regions)
    client_type = random.choice(client_types)
    current_time = start_date_base + timedelta(days=random.randint(0, 180), hours=random.randint(0, 23))
    
    for stage in stages:
        if stage == '1_Document_Submission':
            proc_time = np.random.gamma(shape=2, scale=2)
        elif stage == '2_KYC_AML_Check':
            proc_time = np.random.gamma(shape=4, scale=4.5)
        elif stage == '3_Credit_Risk_Assessment':
            proc_time = np.random.gamma(shape=3, scale=3)
        elif stage == '4_Legal_Compliance_Approval':
            proc_time = np.random.gamma(shape=3.5, scale=4)
        else:
            proc_time = np.random.gamma(shape=1.5, scale=2)

        end_time = current_time + timedelta(hours=proc_time)
        error_prob = 0.22 if stage in ['2_KYC_AML_Check', '4_Legal_Compliance_Approval'] else 0.05
        error_flag = 1 if random.random() < error_prob else 0
        
        data.append({
            'Case_ID': f"CAS-{case_id}",
            'Client_Type': client_type,
            'Region': region,
            'Stage_Name': stage,
            'Start_Timestamp': current_time.strftime('%Y-%m-%d %H:%M:%S'),
            'End_Timestamp': end_time.strftime('%Y-%m-%d %H:%M:%S'),
            'Duration_Hours': round(proc_time, 2),
            'Error_Flag': error_flag
        })
        current_time = end_time

In [18]:
df = pd.DataFrame(data)
df.to_csv('corporate_onboarding_logs.csv', index=False)
print("Successfully generated 'corporate_onboarding_logs.csv' with 25,000 records!")

Successfully generated 'corporate_onboarding_logs.csv' with 25,000 records!


In [20]:
corporate_logs = pd.read_csv("corporate_onboarding_logs.csv")
corporate_logs

,Case_ID,Client_Type,Region,Stage_Name,Start_Timestamp,End_Timestamp,Duration_Hours,Error_Flag
0,CAS-6000,SME,APAC,5_Account_Activation,2026-01-30 21:00:00,2026-01-31 00:07:51,3.13,0
1,CAS-1001,Corporate,EMEA,1_Document_Submission,2026-05-20 12:00:00,2026-05-20 20:52:14,8.87,0
2,CAS-1001,Corporate,EMEA,2_KYC_AML_Check,2026-05-20 20:52:14,2026-05-21 11:39:54,14.79,0
3,CAS-1001,Corporate,EMEA,3_Credit_Risk_Assessment,2026-05-21 11:39:54,2026-05-21 12:32:37,0.88,0
4,CAS-1001,Corporate,EMEA,4_Legal_Compliance_Approval,2026-05-21 12:32:37,2026-05-22 06:27:20,17.91,0
...,...,...,...,...,...,...,...,...
49996,CAS-6000,Institutional,EMEA,1_Document_Submission,2026-03-27 06:00:00,2026-03-27 10:02:47,4.05,0
49997,CAS-6000,Institutional,EMEA,2_KYC_AML_Check,2026-03-27 10:02:47,2026-03-28 08:18:24,22.26,1
49998,CAS-6000,Institutional,EMEA,3_Credit_Risk_Assessment,2026-03-28 08:18:24,2026-03-28 13:54:18,5.60,1
49999,CAS-6000,Institutional,EMEA,4_Legal_Compliance_Approval,2026-03-28 13:54:18,2026-03-29 02:36:25,12.70,0


In [21]:
import sqlite3
import pandas as pd

df = pd.read_csv("corporate_onboarding_logs.csv")
conn = sqlite3.connect(':memory:')
df.to_sql('onboarding_logs', conn, index=False, if_exists='replace')

q1 = """
SELECT 
    Stage_Name,
    COUNT(DISTINCT Case_ID) AS Total_Cases,
    ROUND(AVG(Duration_Hours), 2) AS Avg_Duration_Hours,
    SUM(Error_Flag) AS Total_Errors,
    ROUND(AVG(Error_Flag) * 100, 2) AS Error_Rate_Pct
FROM onboarding_logs
GROUP BY Stage_Name
ORDER BY Stage_Name;
"""
print("--- STAGE PERFORMANCE ---")
print(pd.read_sql_query(q1, conn))

--- STAGE PERFORMANCE ---
                    Stage_Name  Total_Cases  Avg_Duration_Hours  Total_Errors  \
0        1_Document_Submission         5000                3.96           527   
1              2_KYC_AML_Check         5000               18.15          2181   
2     3_Credit_Risk_Assessment         5000                9.05           508   
3  4_Legal_Compliance_Approval         5000               14.01          2249   
4         5_Account_Activation         5000                2.98           487   

   Error_Rate_Pct  
0            5.27  
1           21.81  
2            5.08  
3           22.49  
4            4.87  


In [22]:
# --- PHASE 2: FINANCIAL & BUSINESS RISK CALCULATIONS ---

# 1. Constants & Assumptions
TARGET_SLA_HOURS = 24.0
PENALTY_PER_DELAYED_HOUR = 50.0  # $50/hour SLA breach penalty
REWORK_COST_PER_ERROR = 100.0   # $100/manual error fix

# 2. Extract Stage Metrics from SQLite Connection
df_summary = pd.read_sql_query(q1, conn)

# 3. Calculate Operational Totals
total_cases = df_summary['Total_Cases'].max()  # 5,000 cases
total_avg_duration = df_summary['Avg_Duration_Hours'].sum()  # Total end-to-end cycle time
total_errors_stage_2_4 = df_summary[df_summary['Stage_Name'].str.contains('2_KYC|4_Legal')]['Total_Errors'].sum()

# 4. Financial Calculations
avg_delay_hours = max(0, total_avg_duration - TARGET_SLA_HOURS)
total_sla_breach_hours = total_cases * avg_delay_hours
direct_financial_loss = total_sla_breach_hours * PENALTY_PER_DELAYED_HOUR
rework_cost = total_errors_stage_2_4 * REWORK_COST_PER_ERROR
total_financial_impact = direct_financial_loss + rework_cost

# Display Results
print("=== PHASE 2: FINANCIAL IMPACT SUMMARY ===")
print(f"Total Onboarding Cycle Time: {total_avg_duration:.2f} Hours")
print(f"Average SLA Breach per Case: {avg_delay_hours:.2f} Hours beyond {TARGET_SLA_HOURS}h Target")
print(f"Total SLA Breach Hours:      {total_sla_breach_hours:,.0f} Hours")
print(f"Direct SLA Penalty Exposure: ${direct_financial_loss:,.2f}")
print(f"Rework Cost (Stage 2 & 4):   ${rework_cost:,.2f}")
print(f"Total Business Risk Impact:  ${total_financial_impact:,.2f}")

=== PHASE 2: FINANCIAL IMPACT SUMMARY ===
Total Onboarding Cycle Time: 48.15 Hours
Average SLA Breach per Case: 24.15 Hours beyond 24.0h Target
Total SLA Breach Hours:      120,750 Hours
Direct SLA Penalty Exposure: $6,037,500.00
Rework Cost (Stage 2 & 4):   $443,000.00
Total Business Risk Impact:  $6,480,500.00
